# 00 — Inventory, integrity, and the two decisions that must be made before any training

**Runtime → Change runtime type → CPU.** This notebook trains nothing, and running it on a
GPU just burns compute units.

It exists to settle three things on the record:

1. **What is actually in the release** — counts, resolutions, and whether all 7 experts'
   masks are present for every eye.
2. **The camera/split confound**, re-derived here rather than taken on trust. It is the fact
   that made a camera-versus-convention 2×2 impossible, so it belongs in your own output.
3. **Two frozen decisions**: the vCDR measured from the *official per-expert PNG masks*
   (not the ellipse sheets), and the **crop margin**, chosen once, from the data, before any
   model exists — so it can never be tuned against the result.

About 20 minutes, most of it the one-pass mask scan, which checkpoints as it goes.


In [ ]:
# --- environment -----------------------------------------------------------
# WHERE THINGS LIVE, and why:
#   PROJECT  this project's code (src/, analysis/)   -> Drive, or a GitHub clone
#   RAW      the 1.5 GB dataset                      -> LOCAL disk: Drive I/O is slow,
#                                                       and re-fetching costs ~3 minutes
#   WORK     crops, weights, results                 -> Drive, so a disconnect costs nothing
REPO_URL = "https://github.com/Aduitya9/Opiotica.git"   # cloned fresh each session
DRIVE_PROJECT = "/content/drive/MyDrive/whose-eyes"     # offline fallback if REPO_URL is ""
WORK = "/content/drive/MyDrive/whose-eyes-work"
RAW = "/content/raw"

import json, os, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")

if REPO_URL:
    if not os.path.isdir("/content/project"):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/content/project"], check=True)
    PROJECT = "/content/project"
else:
    PROJECT = DRIVE_PROJECT
assert os.path.isdir(os.path.join(PROJECT, "src")), (
    f"No src/ under {PROJECT}. Copy the `whose-eyes` folder into your Drive at "
    f"{DRIVE_PROJECT}, or set REPO_URL above.")
sys.path.insert(0, PROJECT)
for sub in ("crops", "runs", "results"):
    os.makedirs(os.path.join(WORK, sub), exist_ok=True)
os.makedirs(RAW, exist_ok=True)

# --- GPU ---------------------------------------------------------------------
# Batch size is set from the card, not guessed: U-Net + EfficientNet-B4 at 512 px with AMP
# needs roughly 1.5 GB per sample. Everything else in the schedule stays fixed.
import torch

if torch.cuda.is_available():
    GPU = torch.cuda.get_device_name(0)
    VRAM = torch.cuda.get_device_properties(0).total_memory / 1e9
    BATCH = 24 if VRAM > 60 else 16 if VRAM > 35 else 8 if VRAM > 20 else 4
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
else:
    GPU, VRAM, BATCH = "cpu", 0.0, 2

print(f"project : {PROJECT}")
print(f"work    : {WORK}   (Drive - survives a disconnect)")
print(f"raw     : {RAW}   (local - re-fetched each session)")
print(f"gpu     : {GPU}  {VRAM:.0f} GB  ->  batch size {BATCH}")


In [ ]:
# --- get REFUGE-MultiRater ---------------------------------------------------
# The repo holds ONE 1.5 GB zip, not an unpacked tree, so it is fetched and extracted.
!pip -q install huggingface_hub

from huggingface_hub import hf_hub_download
import glob, zipfile

RELEASE_GLOB = os.path.join(RAW, "**", "Training-400")
if not glob.glob(RELEASE_GLOB, recursive=True):
    zip_path = hf_hub_download(
        repo_id="realslimman/REFUGE-MultiRater",
        filename="REFUGE-Multirater.zip",
        repo_type="dataset",
        local_dir="/content/hf",
    )
    print(f"downloaded {os.path.getsize(zip_path) / 1e9:.2f} GB; extracting...")
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(RAW)
    print("extracted")
else:
    print("already extracted this session")

RELEASE = sorted({os.path.dirname(p) for p in glob.glob(RELEASE_GLOB, recursive=True)})[0]
print("release root:", RELEASE)
print(sorted(os.listdir(RELEASE)))


In [ ]:
# --- inventory: resolution and file completeness, one row per eye ------------
import pandas as pd
from PIL import Image
from src import refuge

INVENTORY = os.path.join(WORK, "results", "inventory.csv")
rows = []
for split, image_id, folder in ([] if os.path.exists(INVENTORY)
                                else refuge.iter_images(RELEASE)):
    files = set(os.listdir(folder))
    jpg = next((f for f in files if f.lower().endswith((".jpg", ".jpeg"))), None)
    width, height = Image.open(os.path.join(folder, jpg)).size if jpg else (0, 0)
    rows.append({
        "split": split, "camera": refuge.CAMERA[split], "image_id": image_id,
        "width": width, "height": height,
        "has_official_cup": f"{image_id}_cup.bmp" in files,
        "has_official_disc": f"{image_id}_disc.bmp" in files,
        "n_expert_cup": sum(f"{image_id}_seg_cup_{k}.png" in files for k in range(1, 8)),
        "n_expert_disc": sum(f"{image_id}_seg_disc_{k}.png" in files for k in range(1, 8)),
    })
if os.path.exists(INVENTORY):          # re-runs are cheap: this takes a few minutes otherwise
    inventory = pd.read_csv(INVENTORY, dtype={"image_id": str})
    print("loaded the saved inventory; delete results/inventory.csv to rebuild it\n")
else:
    inventory = pd.DataFrame(rows)
    # the clinical diagnosis, already resolved in analysis/r1_labels.csv: the release's own
    # REFUGE1*.csv files repeat folder ids across rows, so 5 eyes have conflicting label rows
    # and were dropped. 1,195 eyes keep a clean label; 119 of them are glaucoma.
    clinical = pd.read_csv(os.path.join(PROJECT, "analysis", "r1_labels.csv"),
                           dtype={"folder": str}).rename(columns={"folder": "image_id"})
    inventory = inventory.merge(clinical, on=["split", "image_id"], how="left")
    inventory.to_csv(INVENTORY, index=False)

print(f"{len(inventory)} eyes")
print(f"with a clean clinical label: {int(inventory.label.notna().sum())}  "
      f"(glaucoma {int(inventory.label.fillna(0).sum())})")
incomplete = inventory[(inventory.n_expert_cup < 7) | (inventory.n_expert_disc < 7)
                       | ~inventory.has_official_cup | ~inventory.has_official_disc]
print(f"eyes missing any mask: {len(incomplete)}")
if len(incomplete):
    display(incomplete.head(20))


In [ ]:
# --- the confound, in your own output ----------------------------------------
print("RESOLUTION x SPLIT - if each split has exactly one resolution, camera IS split:\n")
print(pd.crosstab(inventory.split,
                  inventory.width.astype(str) + " x " + inventory.height.astype(str)).to_string())
print("""
Read it carefully. Training-400 is Zeiss at 2124x2056; Validation-400 and Test-400 are
Canon at 1634x1634. No eye appears on both cameras, so camera, split and patient set are the
same variable. A clean camera-versus-convention 2x2 cannot be built from REFUGE, and the
pre-registration treats the cross-dataset result as a LOWER BOUND instead.""")


In [ ]:
# --- one pass over every mask: per-expert vCDR AND disc boxes ----------------
# Fix A2: the training targets are the official per-expert PNGs, so the convention offsets
# must be re-derived from THOSE, not from the ellipse sheets the pilots used.
# Checkpoints every 200 eyes, so a Colab disconnect costs at most a couple of minutes.
import time
import numpy as np
from src import crops as crops_mod

MEASURED = os.path.join(WORK, "results", "png_measurements.csv")
done = pd.read_csv(MEASURED, dtype={"image_id": str}) if os.path.exists(MEASURED) \
    else pd.DataFrame(columns=["image_id"])
seen = set(done.image_id)
records, t0 = [], time.time()

for n, (split, image_id, folder) in enumerate(refuge.iter_images(RELEASE), 1):
    if image_id in seen:
        continue
    rec = {"split": split, "camera": refuge.CAMERA[split], "image_id": image_id}
    for k in range(1, 8):
        cup, disc = refuge.expert_masks(folder, image_id, k)
        ct, cl, cb, cr = crops_mod.mask_box(cup)
        dt, dl, db, dr = crops_mod.mask_box(disc)
        rec[f"cup_v_user{k}"], rec[f"disc_v_user{k}"] = cb - ct, db - dt
        rec[f"box_user{k}"] = f"{dt},{dl},{db},{dr}"
    ocup, odisc = refuge.official_masks(folder, image_id)
    oct_, ocl, ocb, ocr = crops_mod.mask_box(ocup)
    odt, odl, odb, odr = crops_mod.mask_box(odisc)
    rec["cup_v_official"], rec["disc_v_official"] = ocb - oct_, odb - odt
    rec["box_official"] = f"{odt},{odl},{odb},{odr}"
    records.append(rec)
    if len(records) % 200 == 0:
        pd.concat([done, pd.DataFrame(records)]).to_csv(MEASURED, index=False)
        print(f"  {n:4d}/1200   {time.time() - t0:5.0f}s   (checkpointed)")

png = pd.concat([done, pd.DataFrame(records)], ignore_index=True)
for name in [f"user{k}" for k in range(1, 8)] + ["official"]:
    png[f"vcdr_{name}"] = png[f"cup_v_{name}"] / png[f"disc_v_{name}"]
png.to_csv(MEASURED, index=False)
print(f"\n{len(png)} eyes measured in {time.time() - t0:.0f}s")


In [ ]:
# --- do the PNG masks give the same convention offsets as the ellipses? ------
from src import vcdr

RATERS = [f"user{k}" for k in range(1, 8)]
wide = png[[f"vcdr_{r}" for r in RATERS]].rename(columns=lambda c: c.replace("vcdr_", ""))
ellipse = pd.read_csv(os.path.join(PROJECT, "analysis", "r6_official_merge.csv"),
                      dtype={"folder": str})

compare = pd.DataFrame({
    "from PNG masks": vcdr.offsets(wide, RATERS),
    "from ellipse sheets": vcdr.offsets(ellipse, RATERS),
})
compare["difference"] = compare["from PNG masks"] - compare["from ellipse sheets"]
compare["refer@0.6 (PNG)"] = [vcdr.referral_rate(wide[r], 0.6) for r in RATERS]
print(compare.round(4).to_string())

delta_png = float((wide["user1"] - wide["user7"]).mean())
print(f"\nDelta_ann from PNG masks : {delta_png:.4f}")
print(f"Delta_ann in the prereg  : {vcdr.DELTA_ANN:.4f}")
if abs(delta_png - vcdr.DELTA_ANN) > 0.01:
    print("\n*** AMEND THE PRE-REGISTRATION BEFORE RUNNING NOTEBOOK 02. The gate is defined "
          "against Delta_ann, so a shift of more than 0.01 goes into §10 with today's date "
          "and a reason - never silently.")
else:
    print("\nWithin tolerance: the gate keeps Delta_ann = 0.269.")


In [ ]:
# --- decision 2: the crop margin, chosen once, from the data -----------------
# The frozen box must hold the UNION of all seven experts' discs on every eye, or a liberal
# expert's outline gets clipped and the comparison is rigged before training starts.
def column(frame, name):
    return np.array([[int(v) for v in s.split(",")] for s in frame[name]], dtype=float)

m = png.merge(inventory[["image_id", "height", "width"]], on="image_id", how="left")
official = column(m, "box_official")
expert = np.stack([column(m, f"box_user{k}") for k in range(1, 8)])     # (7, N, 4)
union = np.stack([expert[:, :, 0].min(0), expert[:, :, 1].min(0),
                  expert[:, :, 2].max(0), expert[:, :, 3].max(0)], axis=1)
cy = (official[:, 0] + official[:, 2]) / 2
cx = (official[:, 1] + official[:, 3]) / 2
diameter = np.maximum(official[:, 2] - official[:, 0], official[:, 3] - official[:, 1])
H, W = m.height.to_numpy(float), m.width.to_numpy(float)

# The box may extend past the image border; notebook 01 pads those parts. It is NOT clamped
# to the image: clamping collapses the box for a disc near an edge, which capped containment
# at 1190/1200 however large the margin grew. See PREREGISTRATION.md Amendment 2.
results = []
for margin in np.arange(1.2, 3.01, 0.1):
    half = margin * diameter / 2
    held = ((cy - half <= union[:, 0]) & (cx - half <= union[:, 1])
            & (cy + half >= union[:, 2]) & (cx + half >= union[:, 3]))
    outside = np.maximum(0, -(cy - half)) + np.maximum(0, (cy + half) - H) \
        + np.maximum(0, -(cx - half)) + np.maximum(0, (cx + half) - W)
    results.append({"margin": round(float(margin), 2), "contained": int(held.sum()),
                    "fraction": float(held.mean()),
                    "eyes_needing_padding": int((outside > 0).sum())})
margins = pd.DataFrame(results)
print(margins.to_string(index=False))

safe = margins[margins.fraction >= 1.0]
if not len(safe):
    raise SystemExit(
        "\nSTOP. No margin in 1.2-3.0 contains all seven experts' discs on every eye.\n"
        "Do NOT pick the largest and carry on - that silently trades away disc resolution\n"
        "on every crop. Find out which eyes fail and why first; the earlier version of this\n"
        "cell fell back to 3.0 here and it took a code audit to notice.")
MARGIN = float(safe.margin.min())
print(f"\nCHOSEN MARGIN = {MARGIN}  (smallest containing every expert's disc on every eye)")
print(f"eyes needing any padding at this margin: "
      f"{int(margins.loc[margins.margin == MARGIN, 'eyes_needing_padding'].iloc[0])}")

json.dump({"margin": MARGIN, "anchor": "official-merge disc",
           "rule": "smallest margin containing the 7-expert disc union on 100% of eyes",
           "table": results},
          open(os.path.join(WORK, "results", "crop_margin.json"), "w"), indent=2)
print("frozen to results/crop_margin.json - never recomputed later")


## What to record before moving on

| | expected |
|---|---|
| Eyes with all 7 experts + the official merge | **1200** |
| Eyes with a clean clinical label | **1195** (119 glaucoma) |
| Resolutions per split | Zeiss 2124×2056 / Canon 1634×1634 |
| `Delta_ann` from the PNG masks | compare against **0.269**; amend §10 if it moved > 0.01 |
| Chosen crop margin | written to `results/crop_margin.json` |

Any eye missing a mask is **excluded and counted in the paper**, never silently dropped.

Next: `01_frozen_localiser.ipynb` — switch the runtime to **L4** for that one.
